# HSMPO Ground State Optimization with DMRG

This notebook demonstrates how to use DMRG (Density Matrix Renormalization Group) to find ground states of quantum systems using the HSMPO ansatz.

**Key advantages of DMRG:**
- Directly optimizes MPS tensor parameters (not circuit angles)
- Exponentially faster convergence for ground state problems
- Maintains automatic normalization and bond dimension control
- Supports flexible Hamiltonian input formats

In [ ]:
# Setup: Import and create HSMPO
from mpstab import HSMPO
from mpstab.models.ansatze import HardwareEfficient
import numpy as np

# Create a 4-qubit ansatz with 2 layers of hardware-efficient gates
ansatz = HardwareEfficient(nqubits=4, nlayers=2)

# Initialize HSMPO (precomputes MPS once at init)
hsmpo = HSMPO(ansatz=ansatz)

print(f"Number of qubits: {hsmpo.nqubits}")
print(f"Number of circuit parameters: {hsmpo.nparams}")
print(f"MPS structure: {hsmpo.original_circuit_mps}")

In [ ]:
# Example 1: Optimize a single observable
print("=" * 60)
print("Example 1: Optimize Single Observable (ZZZZ)")
print("=" * 60 + "\n")

# Initial expectation value (random circuit)
initial_value = hsmpo.expectation('ZZZZ')
print(f"Initial <ZZZZ>: {initial_value:.6f}")

# Run DMRG to minimize <ZZZZ>
result = hsmpo.minimize_expectation(
    observables='ZZZZ',  # Single Pauli observable
    method='dmrg',       # Use DMRG (default)
    bond_dims=[10, 20],  # Gradual bond dimension growth
    max_sweeps=10,
    verbosity=0
)

# Check results
final_value = hsmpo.expectation('ZZZZ')
print(f"\nDMRG Results:")
print(f"  Ground state energy: {result['energy']:.8f}")
print(f"  Final <ZZZZ>:       {final_value:.8f}")
print(f"  Converged:          {result['converged']}")
print(f"  Sweeps performed:   {result['num_sweeps']}")

In [ ]:
# Example 2: Optimize a weighted combination (dict format)
print("\n" + "=" * 60)
print("Example 2: Optimize Weighted Hamiltonian (Dict Format)")
print("=" * 60 + "\n")

# Reset with new random circuit parameters
hsmpo.set_parameters(np.random.uniform(-np.pi, np.pi, hsmpo.nparams))

# H = ZZZZ + 0.5*XXXX (minimize their weighted sum)
hamiltonian = {
    "ZZZZ": 1.0,   # weight 1.0
    "XXXX": 0.5    # weight 0.5
}

result = hsmpo.minimize_expectation(
    observables=hamiltonian,
    method='dmrg',
    bond_dims=[10, 20, 50],
    max_sweeps=10,
    verbosity=0
)

print(f"Hamiltonian: H = 1.0*<ZZZZ> + 0.5*<XXXX>")
print(f"\nDMRG Results:")
print(f"  Ground state energy: {result['energy']:.8f}")
print(f"  Converged:          {result['converged']}")
print(f"  Sweeps:             {result['num_sweeps']}")

# Verify the individual observables
print(f"\nFinal observables at optimized state:")
print(f"  <ZZZZ>: {hsmpo.expectation('ZZZZ'):.6f}")
print(f"  <XXXX>: {hsmpo.expectation('XXXX'):.6f}")
print(f"  <YYYY>: {hsmpo.expectation('YYYY'):.6f}")

In [ ]:
# Example 3: Optimize using SymbolicHamiltonian (proper Qibo API)
print("\n" + "=" * 60)
print("Example 3: SymbolicHamiltonian (XXZ Model)")
print("=" * 60 + "\n")

from qibo.symbols import X, Y, Z
from qibo.hamiltonians import SymbolicHamiltonian

# Build XXZ Hamiltonian: H = sum_i (X_i X_{i+1} + Y_i Y_{i+1} + 0.5 Z_i Z_{i+1})
ham_form = 0
for i in range(hsmpo.nqubits - 1):
    ham_form += X(i) * X(i + 1)
    ham_form += Y(i) * Y(i + 1)
    ham_form += 0.5 * Z(i) * Z(i + 1)

H_symbolic = SymbolicHamiltonian(nqubits=hsmpo.nqubits, form=ham_form)

# Reset with new parameters
hsmpo.set_parameters(np.random.uniform(-np.pi, np.pi, hsmpo.nparams))

# DMRG automatically handles SymbolicHamiltonian format
result = hsmpo.minimize_expectation(
    observables=H_symbolic,  # Pass SymbolicHamiltonian directly
    method='dmrg',
    bond_dims=[10, 20, 50],
    max_sweeps=10,
    verbosity=0
)

print(f"Hamiltonian type: {type(H_symbolic).__name__}")
print(f"Form: XXZ = sum_i (X_i X_{{i+1}} + Y_i Y_{{i+1}} + 0.5 Z_i Z_{{i+1}})")
print(f"\nDMRG Results:")
print(f"  Ground state energy: {result['energy']:.8f}")
print(f"  Converged:          {result['converged']}")
print(f"  Sweeps:             {result['num_sweeps']}")

## Summary: Supported Hamiltonian Input Formats

The `minimize_expectation()` method accepts multiple formats for flexibility:

| Format | Example | Use Case |
|--------|---------|----------|
| **String** | `"ZZZZ"` | Single observable, simplest case |
| **List** | `["ZZZZ", "XXXX"]` | Multiple observables with equal weight (sum) |
| **Dict** | `{"ZZZZ": 1.0, "XXXX": 0.5}` | Weighted combination, full control |
| **SymbolicHamiltonian** | `H = X(0)*X(1) + ...` | Complex Hamiltonians with proper Qibo API |

All formats are internally converted to a unified representation and solved via DMRG.

## DMRG Configuration Options

```python
result = hsmpo.minimize_expectation(
    observables=...           # Any format above
    method='dmrg',            # Only 'dmrg' is supported
    bond_dims=[10, 20, 50],   # Bond dimension growth per sweep
    cutoff=1e-9,              # SVD truncation cutoff
    tol=1e-6,                 # Energy convergence tolerance
    max_sweeps=10,            # Maximum DMRG sweeps
    verbosity=1               # Print progress (0=silent)
)
```

## Return Value

DMRG returns a dictionary with optimization results:

```python
{
    'ground_state': <optimized MPS>,    # The final state
    'energy': <float>,                  # Ground state energy
    'converged': <bool>,                # Whether DMRG converged
    'num_sweeps': <int>                 # Number of sweeps performed
}
```

After optimization, `hsmpo.original_circuit_mps` is automatically updated with the optimized state.